In [11]:
import os, glob
import pandas as pd
import numpy as np

from matplotlib import pyplot as plt


In [ ]:
data_dir = r"D:\!Work\IFA\!data\Urban_fluxes_NE\processed"

# city_name = 'Tomsk'
# corr_name = 'new_py'

# city_name = 'Beijing's
# corr_name = ''

city_name = 'Moscow'
corr_name = ''


df4sites = {}

data_dir_final = data_dir + '/' + city_name + ('_' if corr_name else '') + corr_name

file_pattern =  data_dir_final + '/*.csv'
display(file_pattern)

for file in glob.glob(file_pattern):
    site = os.path.basename(file).split('.')[0]
    print (file)
    df = pd.read_csv(file, index_col=0, parse_dates=True)
    df4sites[site] = df

'D:\\!Work\\IFA\\!data\\Urban_fluxes_NE\\processed/Moscow/*.csv'

D:\!Work\IFA\!data\Urban_fluxes_NE\processed/Moscow\ERA5.csv
D:\!Work\IFA\!data\Urban_fluxes_NE\processed/Moscow\LOD_31m.csv
D:\!Work\IFA\!data\Urban_fluxes_NE\processed/Moscow\MSU_11m.csv
D:\!Work\IFA\!data\Urban_fluxes_NE\processed/Moscow\MSU_19m.csv
D:\!Work\IFA\!data\Urban_fluxes_NE\processed/Moscow\MSU_2m.csv
D:\!Work\IFA\!data\Urban_fluxes_NE\processed/Moscow\ZNS_56m.csv
D:\!Work\IFA\!data\Urban_fluxes_NE\processed/Moscow\ZNS_56m_new.csv


In [13]:
#filter_vars = ['Qh', 'Qtau'] #, 'Cd']
filter_lims = {'Qh': [-300, 600],
               'Qle': [-300, 600],
               'Qtau': [0, 5],
               'Cd': [0, 1]}
filter_sigmas = {'Qh': 5, 'Qle': 5, 'Qtau': 5, 'Cd': 10}

def apply_qc (df, filter_vars, qc_ext_window = '1h', outl_stat_window = '60D', n_sigma = 5, use_hourly_stats = True):

    df = df.copy()

    for var in filter_vars.keys():

        cur_n_sigma = n_sigma[var] if isinstance (n_sigma, dict) else n_sigma

        if var not in df.columns: continue  
        
        df[f'qc_{var}_lims'] = np.zeros_like(df[var]).astype(bool)
        
        df[f'qc_{var}_lims'] = (df[var] >= filter_lims[var][0]) & (df[var] <= filter_lims[var][1])
        df.loc[np.isnan(df[var]), f'qc_{var}_lims'] = True
        n_lims = (~df[f'qc_{var}_lims']).sum()
        
        df[f'qc_{var}_outl'] = np.ones_like(df[var]).astype(bool)

        cur_n_outl = 1
        n_outl = []

        while cur_n_outl > 0:

            var_f = df[var].where(df[f'qc_{var}_outl'])
            var_f2stats = df[var].where(df[f'qc_{var}_lims'] & df[f'qc_{var}_outl'])

            if use_hourly_stats:
                outl_lim1 = var_f * np.nan
                outl_lim2 = var_f * np.nan
                for hour in range (0, 24):
                    hours_ext = np.arange(hour-1, hour+2)
                    hours_ext [hours_ext<0] += 24

                    is_hours_est = np.isin (df.index.hour, hours_ext)
                    is_hour = df.index.hour == hour

                    var_rol_mean =  var_f2stats.where(is_hours_est).rolling(outl_stat_window, center=True, min_periods=1).mean()
                    var_rol_std  =  var_f2stats.where(is_hours_est).rolling(outl_stat_window, center=True, min_periods=1).std()
            
                    outl_lim1[is_hour] = (var_rol_mean - cur_n_sigma * var_rol_std)[is_hour]
                    outl_lim2[is_hour] = (var_rol_mean + cur_n_sigma * var_rol_std)[is_hour]
            else:
                var_rol_mean =  var_f2stats.rolling(outl_stat_window, center=True, min_periods=1).mean()
                var_rol_std  =  var_f2stats.rolling(outl_stat_window, center=True, min_periods=1).std()
            
                outl_lim1 = (var_rol_mean - cur_n_sigma * var_rol_std)
                outl_lim2 = (var_rol_mean + cur_n_sigma * var_rol_std)

            df[f'{var}_outl_lim1'] = outl_lim1
            df[f'{var}_outl_lim2'] = outl_lim2

            is_outlier = (var_f < outl_lim1) | (var_f > outl_lim2)
            is_outlier [np.isnan(df[var])] = False
            cur_n_outl = (is_outlier).sum()

            df.loc[is_outlier, f'qc_{var}_outl'] = False

            n_outl.append(cur_n_outl)

            if len(n_outl) > 2 and n_outl[-1] <= 2 and n_outl[-2] <= 2:
                break

        print (f'{var}: {n_lims}, {n_outl}')
        df[f'qc_{var}_sum'] = df[[c for c in df.columns if f'qc_{var}' in c]].astype(float).sum(axis=1)
    

    df[f'qc_tot'] = (df[[f'qc_{var}_sum' for var in filter_vars.keys() if var in df.columns]] >= 2).min(axis=1) #in filter_vars.keys()]
    df[f'qc_tot_noCd'] = (df[[f'qc_{var}_sum' for var in filter_vars.keys() if var in df.columns if 'Cd' not in var]] >= 2).min(axis=1) #in filter_vars.keys()]

    print (f'total: { (~df[f'qc_tot']).sum()}, total_noCd: {(~df[f'qc_tot_noCd']).sum()}')

    df[f'qc_tot_ext'] = df[f'qc_tot'].rolling(qc_ext_window, center=True).min().astype(bool)        
    df[f'qc_tot_noCd_ext'] = df[f'qc_tot_noCd'].rolling(qc_ext_window, center=True).min().astype(bool)                
    
    return df

df4sites_qc_d = {}
df4sites_qc_h = {}

for site in df4sites.keys():
    df4sites[site]['Cd'] = (df4sites[site]['Ustar'] / df4sites[site]['Wind_vel']) **2
    #df4sites_qc_d[site] = apply_qc(df4sites[site], filter_lims, use_hourly_stats=False)
    print (f'*** starting QC for {site} ***')
    if site != 'ERA5':
        df4sites_qc_h[site] = apply_qc(df4sites[site], filter_lims, use_hourly_stats=True, n_sigma=filter_sigmas)
    else:
        df4sites_qc_h[site] = df4sites[site].copy()
    #ak



*** starting QC for ERA5 ***
*** starting QC for LOD_31m ***
Qh: 324, [22, 5, 0]
Qtau: 3536, [41, 8, 0]
Cd: 1024, [262, 0]
total: 4497, total_noCd: 3768
*** starting QC for MSU_11m ***
Qh: 0, [10, 1, 0]
Qtau: 45, [110, 19, 5, 2, 1]
Cd: 195, [188, 2, 0]
total: 356, total_noCd: 146
*** starting QC for MSU_19m ***
Qh: 0, [6, 1, 0]
Qtau: 0, [22, 3, 2, 0]
Cd: 21, [44, 3, 2, 0]
total: 83, total_noCd: 34
*** starting QC for MSU_2m ***
Qh: 5, [24, 1, 0]
Qtau: 0, [28, 9, 1, 0]
Cd: 904, [189, 0]
total: 946, total_noCd: 60
*** starting QC for ZNS_56m ***
Qh: 3, [42, 9, 1, 1]
Qtau: 0, [87, 34, 10, 3, 1, 3, 3, 1, 0]
Cd: 63, [138, 19, 10, 0]
total: 346, total_noCd: 185
*** starting QC for ZNS_56m_new ***
Qh: 3, [42, 9, 1, 1]
Qtau: 0, [87, 34, 10, 3, 1, 3, 3, 1, 0]
Cd: 63, [138, 19, 10, 0]
total: 346, total_noCd: 185


In [14]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
pd.options.plotting.backend = "plotly"

if city_name == 'Tomsk':

    vars2draw = ['Qh', 'Qtau', 'Cd']
    site2draw = 'Oblkom_30m'


    df2draw = pd.DataFrame(df4sites_qc_h[site2draw][vars2draw])
    df2draw_f = df2draw.where(df4sites_qc_h[site2draw]['qc_tot'] > 0)

    # df2draw = df2draw.resample('1D').mean()
    # df2draw_f = df2draw_f.resample('1D').mean()


    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04, 
    )

    fig.add_trace(
        go.Scatter(x=df2draw.index, y=df2draw['Qh'], name='Qh (raw)', mode='lines',  line=dict(width=2)), #, line=dict(color='blue')),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(x=df2draw_f.index, y=df2draw_f['Qh'], name='Qh (qc)', mode='lines',  line=dict(width=2)), #, line=dict(color='red')),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(x=df2draw.index, y=df2draw['Qtau'], name='Qtau (raw)', mode='lines',  line=dict(width=2)), #, line=dict(color='blue')),
        row=2, col=1
    )

    fig.add_trace(
        go.Scatter(x=df2draw_f.index, y=df2draw_f['Qtau'], name='Qtau (qc)', mode='lines',  line=dict(width=2)), #, line=dict(color='red')),
        row=2, col=1
    )

    fig.add_trace(
        go.Scatter(x=df2draw.index, y=df2draw['Cd'], name='Cd (raw)', mode='lines',  line=dict(width=2)), #, line=dict(color='blue')),
        row=3, col=1
    )

    fig.add_trace(
        go.Scatter(x=df2draw_f.index, y=df2draw_f['Cd'], name='Cd (qc)', mode='lines',  line=dict(width=2)), #, line=dict(color='red')),
        row=3, col=1
    )

    fig.update_xaxes(range=['2025-04-01', '2025-05-01'])
    fig.update_yaxes(title_text='Qh (W/m²)', range=[-200, 500], row=1, col=1)
    fig.update_yaxes(title_text='Qtau (N/m²)', range=[0, 4], row=2, col=1)
    fig.update_yaxes(title_text='Cd', range=[0, 0.2], row=3, col=1)

    #fig.update_xaxes(range=['2024-06-01', '2025-07-01'])
    # fig.update_yaxes(title_text='Qh (W/m²)', range=[-50, 200], row=1, col=1)
    # fig.update_yaxes(title_text='Qtau (N/m²)', range=[0, 1], row=2, col=1)
    # fig.update_yaxes(title_text='Cd', range=[0, 0.1], row=3, col=1)


    fig.update_layout(
        width=800, height = 600,
        margin=dict(l=10, r=10, t=10, b=10),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.1,
            )
    )
    fig.show()


    fig.write_image('qc_plot.png', scale=4)


In [15]:
out_dir = data_dir_final + '/L1'
os.makedirs(out_dir, exist_ok=True)

for site in df4sites_qc_h.keys():

    df4sites_qc_h[site] = df4sites_qc_h[site].drop(columns = [c for c in df4sites_qc_h[site].columns if ('qc' in c and 'tot' not in c) or 'lim' in c])

    for c in [c for c in df4sites_qc_h[site].columns if 'qc_' in c]:
        df4sites_qc_h[site][c] = df4sites_qc_h[site][c].astype(int)
        
    df4sites_qc_h[site].to_csv(f'{out_dir}/{site}.csv')